# 5. Fine-tune F5-TTS on the teacher's voice

**What this notebook does (run the cells top to bottom, one at a time):**
1. Checks the GPU and your files.  2. Installs F5-TTS.  3. Prepares your dataset.  4. **Fine-tunes** the model.
5. Scores every saved checkpoint against the teacher's held-out audio and against the *un-fine-tuned* model.
6. Lets you listen, pick the best one, and saves it to your Drive.

**Before you start**
- Runtime menu -> *Change runtime type* -> **GPU** (T4 is fine).
- Get these 2 files into Colab. EITHER drag them into the **Files panel on the left** of this page (easiest; they are deleted when the Colab session ends, so keep the originals on your Mac), OR put them in a Drive folder called `voice_capstone`:
  - `dataset_for_training.zip` (made by `python prepare_dataset.py build`, in your `dataset` folder)
  - `held_out.zip` (the `held_out` folder zipped; on the Mac terminal: `cd dataset && zip -r held_out.zip held_out`)
- Step 1 asks to connect Google Drive (click *Allow*): the results and the final model are saved there so they survive when Colab resets. If Drive refuses to connect, the notebook carries on without it and downloads the final model to your computer at the end (Step 8).
- Keep this browser tab open while it trains. If Colab asks you to restart the runtime after the install cell, do it and run again from the config cell.

**Licence note:** the pretrained F5-TTS weights are CC-BY-NC (non-commercial). Fine for a college capstone; do not sell the output.

## Step 0 - settings (the only cell you may need to edit)

In [ ]:
# ============================ CONFIG ============================
DRIVE_FOLDER = "voice_capstone"        # folder in My Drive where results are saved (and where the zips can be, if not in the Files panel)
TRAIN_ZIP    = "dataset_for_training.zip"
HELD_OUT_ZIP = "held_out.zip"
RUN_NAME     = "teacher"               # letters, digits and _ only

# Training. 4000 updates is about 100 passes (epochs) over ~24 minutes of audio.
TARGET_UPDATES = 4000                  # total training updates (the notebook works out the epochs)
SAVE_EVERY     = 800                   # keep a checkpoint every N updates so you can compare early vs late
LEARNING_RATE  = 1e-5                  # the F5-TTS authors' fine-tuning default
BATCH_FRAMES   = 3200                  # the authors' default; fits a 16 GB T4 with fp16
WARMUP_UPDATES = 100

# Scoring / listening
EVAL_SENTENCES = 10                    # how many held-out sentences each checkpoint must speak
NFE_STEPS      = 32                    # denoising steps at generation time (32 = the authors' default)
SEED           = 1234
REF_CLIP       = "auto"                # "auto", or a training clip name such as "L0_0042.wav" (5-8 s, clean, no stumbles)
WHISPER_MODEL  = "small.en"            # listens to the generated speech to measure word errors

# Extra sentences in the teacher's domain that the teacher never said (like your real summaries will be).
EXTRA_SENTENCES = [
    "In a format string attack, the percent N specifier writes the number of characters printed so far to an address.",
    "The summary of this lecture covers the stack layout, the return address, and why user input must never be used as a format string.",
    "First, the attacker finds the offset of the buffer on the stack. Then, they place the target address at the start of the input. Finally, they use percent N to overwrite the value at that address.",
]
# ================================================================

F5_COMMIT = "600e5fdfbf319f84735557aca80a7aaf60d487e6"   # the exact F5-TTS version this notebook was checked against
assert RUN_NAME.replace("_", "").isalnum(), "RUN_NAME must contain only letters, digits and _"

## Step 1 - GPU, disk, files

In [ ]:
import os, sys, subprocess, shutil, json, math, re, glob, zipfile, time, io, contextlib, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path(os.environ.get("NB_ROOT", "/content"))           # where everything is built (Colab: /content)
USE_CPU = bool(os.environ.get("NB_CPU"))                     # only for testing on a laptop; never set this on Colab


def run_stream(cmd, cwd=None, env=None):
    """Run a command and show its output live in the notebook."""
    e = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    if env:
        e.update(env)
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while True:
        chunk = os.read(p.stdout.fileno(), 4096)
        if not chunk:
            break
        sys.stdout.write(chunk.decode("utf-8", "replace"))
        sys.stdout.flush()
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(str(c) for c in cmd)}")


# Google Drive
DRIVE_OK = True
if "NB_DRIVE" in os.environ:
    DRIVE = Path(os.environ["NB_DRIVE"])
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    except Exception as ex:
        DRIVE_OK = False
        DRIVE = ROOT / "no_drive"
        print("\nGoogle Drive could NOT be connected:", str(ex)[:100])
        print("Continuing WITHOUT Drive. Results go to", DRIVE / "f5_run", "and are DELETED when Colab resets.")
        print("Step 8 will download the final model to your computer, so nothing is lost if you let it finish.\n")
OUT = DRIVE / "f5_run"
OUT.mkdir(parents=True, exist_ok=True)

# GPU
if not USE_CPU:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    assert r.returncode == 0 and r.stdout.strip(), (
        "No GPU found. In the menu: Runtime -> Change runtime type -> choose a GPU (T4), then run this cell again.")
    print("GPU:", r.stdout.strip())

# Disk: every checkpoint is ~5.4 GB (model + EMA copy + optimizer state)
free_gb = shutil.disk_usage(ROOT).free / 1e9
n_ckpt = max(1, TARGET_UPDATES // SAVE_EVERY)
need_gb = n_ckpt * 5.5 + 5.5 + 12
KEEP_LAST_N = -1 if free_gb >= need_gb else 3
print(f"Free disk: {free_gb:.0f} GB. Keeping {'all' if KEEP_LAST_N == -1 else 'only the last 3'} checkpoints (needs ~{need_gb:.0f} GB for all).")

# Files
ZIP_PATHS = {}
for f in (TRAIN_ZIP, HELD_OUT_ZIP):
    # look in the Colab Files panel (the top level and one folder down, e.g. sample_data), then in your Drive
    places = [ROOT] + sorted(d for d in ROOT.glob("*/") if d.is_dir() and d.name not in ("drive", "F5-TTS", "no_drive"))
    places += [DRIVE, DRIVE.parent]
    found = [d / f for d in places if (d / f).exists()]
    assert found, (f"Not found: {f}\nDrag it into the Files panel on the left of Colab "
                   f"(or put it in My Drive/{DRIVE_FOLDER}), then run this cell again.")
    ZIP_PATHS[f] = found[0]
print("Using these files:", *ZIP_PATHS.values(), sep="\n  ")

## Step 2 - install F5-TTS (about 3-5 minutes the first time)

In [ ]:
import importlib.util

REPO = ROOT / "F5-TTS"
if not REPO.exists():
    run_stream(["git", "clone", "-q", "https://github.com/SWivid/F5-TTS.git", REPO])
    run_stream(["git", "-C", REPO, "checkout", "-q", F5_COMMIT])


def patch(path, old, new):
    s = Path(path).read_text()
    if new in s:
        return "already patched"
    assert old in s, f"Could not patch {path}: the F5-TTS source is not the expected version."
    Path(path).write_text(s.replace(old, new))
    return "patched"


# 1) Colab has only 2 CPU cores: the default of 16 data-loading workers can run it out of RAM.
print(patch(REPO / "src/f5_tts/model/trainer.py",
            "def train(self, train_dataset: Dataset, num_workers=16,", "def train(self, train_dataset: Dataset, num_workers=2,"))
# 2) Read wav files with soundfile instead of torchaudio.load (newer torchaudio needs an extra library that often fails to load).
print(patch(REPO / "src/f5_tts/model/dataset.py",
            "audio, source_sample_rate = torchaudio.load(audio_path)",
            'audio_np, source_sample_rate = __import__("soundfile").read(audio_path, dtype="float32", always_2d=True)\n'
            "            audio = torch.from_numpy(audio_np.T.copy())"))
print(patch(REPO / "src/f5_tts/infer/utils_infer.py",
            "audio, sr = torchaudio.load(ref_audio)",
            'audio_np, sr = __import__("soundfile").read(ref_audio, dtype="float32", always_2d=True)\n'
            "    audio = torch.from_numpy(audio_np.T.copy())"))

if importlib.util.find_spec("f5_tts") is None:
    run_stream([sys.executable, "-m", "pip", "install", "-q", "-e", REPO])
extras = {"num2words": "num2words", "speechbrain": "speechbrain", "jiwer": "jiwer", "faster_whisper": "faster-whisper"}
missing = [pip_name for mod, pip_name in extras.items() if importlib.util.find_spec(mod) is None]
if missing:
    run_stream([sys.executable, "-m", "pip", "install", "-q", *missing])
importlib.invalidate_caches()
sys.path.insert(0, str(REPO / "src"))
print("F5-TTS ready.")

## Step 3 - prepare the dataset
This unzips your files, writes any digits as words (so the text matches how the teacher says them), drops clips with symbols the model
cannot read, and converts everything into the format F5-TTS trains on. It then tells you the training plan.

In [ ]:
from num2words import num2words
import soundfile as sf

DATA_IN = ROOT / "data_in"
shutil.rmtree(DATA_IN, ignore_errors=True)
INPUTS = ROOT / "inputs"
INPUTS.mkdir(exist_ok=True)
for z, d in ((TRAIN_ZIP, "train"), (HELD_OUT_ZIP, "held_out")):
    local = INPUTS / z                       # work from a copy on Colab's own disk: reading big files straight off Drive can fail
    for attempt in range(4):
        try:
            shutil.copyfile(ZIP_PATHS[z], local)
            with zipfile.ZipFile(local) as zf:
                bad = zf.testzip()
                assert bad is None, f"{z} is damaged (first bad file: {bad})"
                zf.extractall(DATA_IN / d)
            break
        except (OSError, zipfile.BadZipFile, AssertionError) as ex:
            print(f"  {z}: attempt {attempt + 1} failed ({str(ex)[:90]})")
            shutil.rmtree(DATA_IN / d, ignore_errors=True)
            if attempt == 3:
                raise RuntimeError(f"{z} could not be read. Re-upload it into the Colab Files panel (the file may be damaged "
                                   f"or the upload incomplete), delete the old copy, and run Step 1 again.")
            time.sleep(5)
    print(f"  {z}: copied and unzipped OK")
train_csv = next(p for p in (DATA_IN / "train").rglob("train.csv") if "__MACOSX" not in str(p))
held_csv = next(p for p in (DATA_IN / "held_out").rglob("held_out.csv") if "__MACOSX" not in str(p))
TRAIN_DIR, HELD_DIR = train_csv.parent, held_csv.parent


def read_rows(path, sep="|"):
    lines = Path(path).read_text(encoding="utf-8-sig").splitlines()
    return [l.split(sep, 1) for l in lines[1:] if sep in l]


# characters the pretrained model knows
VOCAB_FILE = REPO / "data/Emilia_ZH_EN_pinyin/vocab.txt"
vocab = set(line.rstrip("\n") for line in open(VOCAB_FILE, encoding="utf-8"))
TRANS = str.maketrans({";": ",", "“": '"', "”": '"', "‘": "'", "’": "'"})   # the same substitutions F5-TTS applies


def to_train_text(t):
    t = " ".join(t.replace("|", " ").split())
    t = re.sub(r"\b\d+\b", lambda m: num2words(int(m.group())), t)          # "0 X" -> "zero X", "1 hour" -> "one hour"
    return t


kept, changed, dropped = [], [], []
for wav, text in read_rows(train_csv):
    new = to_train_text(text)
    bad = sorted({c for c in new.translate(TRANS) if c not in vocab})
    if bad:
        dropped.append((wav, new, bad))
        continue
    if new != text:
        changed.append((text, new))
    kept.append((str((TRAIN_DIR / wav).resolve()), new))

print(f"{len(kept)} clips kept, {len(dropped)} dropped (characters the model cannot read), {len(changed)} texts rewritten.")
for old, new in changed:
    print("  rewritten:", old, "\n         ->", new)
for wav, new, bad in dropped:
    print("  dropped:", wav, "unknown characters:", bad)
assert len(kept) >= 20, "Too few usable clips. Check your dataset zip."

(ROOT / "train_abs.csv").write_text("audio_file|text\n" + "".join(f"{a}|{t}\n" for a, t in kept), encoding="utf-8")
data_dir = REPO / "data" / f"{RUN_NAME}_pinyin"
shutil.rmtree(data_dir, ignore_errors=True)
run_stream([sys.executable, "src/f5_tts/train/datasets/prepare_csv_wavs.py", ROOT / "train_abs.csv", data_dir], cwd=REPO)

# training plan: replicate F5-TTS's own batching to know how many updates make one epoch
durations = json.load(open(data_dir / "duration.json"))["duration"]
frames = sorted(d * 24000 / 256 for d in durations)
assert max(frames) <= BATCH_FRAMES, "A clip is longer than one batch; raise BATCH_FRAMES."
batches, cur, n = 0, 0.0, 0
for fl in frames:
    if cur + fl <= BATCH_FRAMES and n < 64:
        cur, n = cur + fl, n + 1
    else:
        batches += n > 0
        cur, n = fl, 1
batches += n > 0
EPOCHS = max(1, math.ceil(TARGET_UPDATES / batches))
TOTAL_UPDATES = EPOCHS * batches
print(f"\nTRAINING PLAN: {len(durations)} clips, {sum(durations) / 60:.1f} minutes of audio")
print(f"  {batches} updates per epoch x {EPOCHS} epochs = {TOTAL_UPDATES} updates")
print(f"  a numbered checkpoint every {SAVE_EVERY} updates: {list(range(SAVE_EVERY, TOTAL_UPDATES + 1, SAVE_EVERY))}")
assert WARMUP_UPDATES < TOTAL_UPDATES // 2

# held-out set (never trained on): audio + text
held = [(str((HELD_DIR / w).resolve()), to_train_text(t)) for w, t in read_rows(held_csv)]
print(f"Held-out test clips: {len(held)}")

## Step 4 - FINE-TUNE (this is the long cell)
Watch the progress bar: it shows the speed and the time remaining. `loss` should drift down and then flatten (it is noisy).
If Colab disconnects, run Step 0-3 again and then this cell: it resumes from the last saved point if the files are still there.

In [ ]:
ckpt_dir = REPO / "ckpts" / RUN_NAME
cmd = [sys.executable, "-m", "accelerate.commands.launch", "--num_processes", "1"]
cmd += ["--cpu"] if USE_CPU else ["--mixed_precision=fp16"]
cmd += [REPO / "src/f5_tts/train/finetune_cli.py",
        "--exp_name", "F5TTS_v1_Base", "--dataset_name", RUN_NAME, "--finetune",
        "--learning_rate", LEARNING_RATE, "--batch_size_per_gpu", BATCH_FRAMES, "--batch_size_type", "frame",
        "--max_samples", 64, "--epochs", EPOCHS, "--num_warmup_updates", WARMUP_UPDATES,
        "--save_per_updates", SAVE_EVERY, "--last_per_updates", min(100, SAVE_EVERY),
        "--keep_last_n_checkpoints", KEEP_LAST_N, "--tokenizer", "pinyin"]
t0 = time.time()
run_stream(cmd, cwd=REPO)
print(f"\nTraining finished in {(time.time() - t0) / 60:.1f} minutes.")
for p in sorted(ckpt_dir.glob("*.pt")) + sorted(ckpt_dir.glob("*.safetensors")):
    print(f"  {p.name:40s} {p.stat().st_size / 1e9:5.2f} GB")

## Step 5 - shrink the checkpoints (5.4 GB -> 0.7 GB each) and make a safety copy on Drive
Each training checkpoint holds the optimizer state, which is only needed for training. For talking we only need the weights.
For each checkpoint we keep two versions: **ema** (a smoothed average, the usual choice) and **raw** (the latest weights).

In [ ]:
import torch, gc
from safetensors.torch import save_file

SLIM = ROOT / "slim"
SLIM.mkdir(exist_ok=True)


def to_half(sd):
    return {k: (v.detach().half() if v.is_floating_point() else v.detach()).contiguous().clone() for k, v in sd.items()}


def slim(ckpt):
    ema_p, raw_p, upd_p = SLIM / f"{ckpt.stem}_ema.safetensors", SLIM / f"{ckpt.stem}_raw.safetensors", SLIM / f"{ckpt.stem}.update"
    if not (ema_p.exists() and raw_p.exists() and upd_p.exists()):
        ck = torch.load(ckpt, map_location="cpu", weights_only=True)
        save_file(to_half(ck["ema_model_state_dict"]), str(ema_p))
        save_file(to_half(ck["model_state_dict"]), str(raw_p))
        upd_p.write_text(str(ck.get("update")))
        del ck
        gc.collect()
        print(f"  {ckpt.name}: update {upd_p.read_text()} -> {ema_p.name}, {raw_p.name}")
    return ema_p, raw_p


def ckpt_update(p):
    return 10**9 if p.stem == "model_last" else int(p.stem.split("_")[1])


ckpts = sorted((p for p in ckpt_dir.glob("model_*.pt")), key=ckpt_update)
assert ckpts, f"No checkpoints found in {ckpt_dir}. Did training finish?"
for p in ckpts:
    slim(p)
UPD = {p.stem: int((SLIM / f"{p.stem}.update").read_text()) for p in ckpts}
# model_last.pt is often the same moment as the newest numbered checkpoint: do not score the same weights twice
numbered = {UPD[p.stem] for p in ckpts if p.stem != "model_last"}
ckpts = [p for p in ckpts if not (p.stem == "model_last" and UPD[p.stem] in numbered)]
print("Checkpoints to score:", [f"{p.stem} (update {UPD[p.stem]})" for p in ckpts])

last_ema, last_raw = slim(ckpt_dir / "model_last.pt")   # always keep a safety copy of the newest weights
(OUT / "slim").mkdir(exist_ok=True)
for p in (last_ema, last_raw):
    shutil.copy(p, OUT / "slim" / p.name)
shutil.copy(data_dir / "vocab.txt", OUT / "vocab.txt")
print("Safety copy of the last checkpoint saved to", OUT / "slim")

## Step 6 - score every checkpoint, and the model *before* fine-tuning
For each model the notebook makes it speak the same sentences, using a short clip of the teacher as the voice prompt, and measures:
- **speaker_sim** - how close the voice is to the teacher's REAL held-out recordings (higher is better)
- **wer** - word error rate: a speech recogniser listens to the output and compares with the intended text (lower is better)
- **utmos** - predicted naturalness, about 1 to 5 (higher is better)

The `baseline` row is the pretrained model with NO fine-tuning, so you can see what fine-tuning bought you.
The `real_teacher` row is the held-out recordings themselves: the ceiling you cannot beat.

In [ ]:
import numpy as np, pandas as pd, torchaudio, librosa, jiwer
from omegaconf import OmegaConf
from cached_path import cached_path
from IPython.display import Audio, display

DEVICE = "cpu" if USE_CPU else "cuda"
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process, preprocess_ref_audio_text
from f5_tts.model import DiT

ARCH = OmegaConf.to_container(OmegaConf.load(REPO / "src/f5_tts/configs/F5TTS_v1_Base.yaml").model.arch)
VOCAB = str(data_dir / "vocab.txt")
vocoder = load_vocoder(vocoder_name="vocos", device=DEVICE)

# ---- the voice prompt: one clean training clip ----
if REF_CLIP == "auto":
    cands = [(a, t) for a, t in kept if 5.5 <= sf.info(a).duration <= 7.5 and 60 <= len(t) <= 140 and t[-1] == "."]
    cands = cands or [(a, t) for a, t in kept if 4 <= sf.info(a).duration <= 9]
    ref_wav, ref_text = sorted(cands, key=lambda c: abs(sf.info(c[0]).duration - 6.5))[0]
else:
    ref_wav, ref_text = next((a, t) for a, t in kept if Path(a).name == REF_CLIP)
print("Voice prompt:", Path(ref_wav).name, "|", ref_text)
display(Audio(ref_wav))
ref_audio_proc, ref_text_proc = preprocess_ref_audio_text(ref_wav, ref_text, show_info=lambda *a, **k: None)

# ---- sentences to speak ----
held_texts = [t for _, t in held][:EVAL_SENTENCES]
texts = held_texts + list(EXTRA_SENTENCES)
n_held = len(held_texts)
print(f"{n_held} held-out sentences (real teacher audio exists) + {len(EXTRA_SENTENCES)} extra sentences")

# ---- scoring tools ----
def norm(t):
    return " ".join(re.sub(r"[^a-z0-9' ]", " ", t.lower()).split())


def load_wave(path, sr_target=16000):
    data, fs = sf.read(str(path), dtype="float32", always_2d=True)
    sig = torch.from_numpy(data.T)
    if fs != sr_target:
        sig = torchaudio.functional.resample(sig, fs, sr_target)
    return sig.mean(0, keepdim=True)


embed = None
try:
    try:
        from speechbrain.inference.speaker import EncoderClassifier
    except ImportError:
        from speechbrain.pretrained import EncoderClassifier
    clf = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir=str(ROOT / "ecapa_model"),
                                         run_opts={"device": DEVICE})

    def embed(path):
        with torch.no_grad():
            return clf.encode_batch(load_wave(path).to(DEVICE)).squeeze().cpu()
except Exception as ex:
    print("Speaker similarity is unavailable:", ex)

from faster_whisper import WhisperModel
whisper = None
for dev, ctype, name in ((("cuda", "float16", WHISPER_MODEL),) if not USE_CPU else ()) + (("cpu", "int8", WHISPER_MODEL),):
    try:
        whisper = WhisperModel(name, device=dev, compute_type=ctype)
        list(whisper.transcribe(np.zeros(16000, dtype=np.float32), language="en")[0])    # test run
        print(f"Word-error checker: {name} on {dev}")
        break
    except Exception as ex:
        print(f"Whisper on {dev} failed ({str(ex)[:80]}); trying the next option")
        whisper = None
assert whisper is not None, "Could not start Whisper."


def transcribe(path):
    segs, _ = whisper.transcribe(str(path), language="en")
    return " ".join(s.text.strip() for s in segs)


utmos = None
try:
    utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)
    utmos = utmos.to(DEVICE)
except Exception as ex:
    print("Naturalness score (UTMOS) is unavailable:", ex)


def score(path, text, centroid, own_emb=None):
    row = {"speaker_sim": np.nan, "wer": np.nan, "utmos": np.nan}
    try:
        if embed is not None:
            row["speaker_sim"] = torch.nn.functional.cosine_similarity(own_emb if own_emb is not None else embed(path), centroid, dim=0).item()
    except Exception as ex:
        print("speaker_sim failed:", ex)
    try:
        row["wer"] = jiwer.wer(norm(text), norm(transcribe(path)))
    except Exception as ex:
        print("wer failed:", ex)
    try:
        if utmos is not None:
            wave, sr = librosa.load(str(path), sr=None, mono=True)
            with torch.no_grad():
                row["utmos"] = float(utmos(torch.from_numpy(wave).unsqueeze(0).to(DEVICE), sr).item())
    except Exception as ex:
        print("utmos failed:", ex)
    return row


# ---- the real teacher (held-out) ----
rows = []
held_embs = [embed(a) for a, _ in held] if embed is not None else []
centroid = torch.stack(held_embs).mean(0) if held_embs else None
if held_embs and len(held_embs) > 1:
    for i, (a, t) in enumerate(held[:n_held]):
        others = torch.stack([e for j, e in enumerate(held_embs) if j != i]).mean(0)       # leave one out: fair
        rows.append({"variant": "real_teacher", "idx": i, "sentence": t, **score(a, t, others, own_emb=held_embs[i])})
print("Scored the real held-out recordings.")

# ---- generation ----
EVAL_DIR = ROOT / "eval"
shutil.rmtree(EVAL_DIR, ignore_errors=True)


class _NoBar:                      # silences F5-TTS's per-sentence progress bars
    @staticmethod
    def tqdm(iterable, *a, **k):
        return iterable


def generate(model, text, out_path):
    torch.manual_seed(SEED)
    with contextlib.redirect_stdout(io.StringIO()):
        wave, sr, _ = infer_process(ref_audio_proc, ref_text_proc, text, model, vocoder, mel_spec_type="vocos",
                                    nfe_step=NFE_STEPS, device=DEVICE, show_info=lambda *a, **k: None, progress=_NoBar)
    sf.write(str(out_path), wave, sr)


def eval_variant(label, ckpt_path, use_ema):
    model = load_model(DiT, ARCH, str(ckpt_path), mel_spec_type="vocos", vocab_file=VOCAB, use_ema=use_ema, device=DEVICE)
    d = EVAL_DIR / label
    d.mkdir(parents=True, exist_ok=True)
    for i, t in enumerate(texts):
        p = d / f"{i:02d}.wav"
        generate(model, t, p)
        rows.append({"variant": label, "idx": i, "sentence": t, **score(p, t, centroid)})
    del model
    gc.collect()
    if not USE_CPU:
        torch.cuda.empty_cache()
    print(f"  done: {label}")


t0 = time.time()
BASELINE = str(cached_path("hf://SWivid/F5-TTS/F5TTS_v1_Base/model_1250000.safetensors"))
eval_variant("baseline_no_finetune", BASELINE, True)
for p in ckpts:
    tag = f"step{UPD[p.stem]}"
    ema_p, raw_p = SLIM / f"{p.stem}_ema.safetensors", SLIM / f"{p.stem}_raw.safetensors"
    eval_variant(f"{tag}_ema", ema_p, True)
    eval_variant(f"{tag}_raw", raw_p, False)
print(f"Generated and scored in {(time.time() - t0) / 60:.1f} minutes.")

df = pd.DataFrame(rows)
df.to_csv(OUT / "results_all_clips.csv", index=False)
summary = df.groupby("variant")[["speaker_sim", "wer", "utmos"]].mean().round(3)
order = (["real_teacher"] if "real_teacher" in summary.index else []) + ["baseline_no_finetune"] + \
        [v for v in summary.index if v not in ("real_teacher", "baseline_no_finetune")]
summary = summary.loc[order]
summary.to_csv(OUT / "results_summary.csv")
print("\nRESULTS (average over the sentences)\n")
print(summary.to_string())

**How to read the table**
- `speaker_sim` going up from `baseline_no_finetune` to the later checkpoints means the voice is getting closer to the teacher.
- If `wer` shoots up or `utmos` drops in the latest checkpoints, the model is over-training on 24 minutes of audio (it starts to mumble). Prefer an earlier checkpoint then.
- `*_ema` vs `*_raw`: the F5-TTS authors warn that the smoothed `ema` weights can lag behind early in fine-tuning, so both are shown.
- The numbers do not measure **accent**. Use your ears for that (next cell).

## Step 7 - listen (this is the real test)

In [ ]:
# The same sentence, the real teacher first, then the un-fine-tuned model, then every checkpoint.
SENTENCE_IDX = [0, 3]                       # which held-out sentences to listen to (0 .. EVAL_SENTENCES-1)
TECH_IDX = n_held                           # first extra sentence (technical)
variants = [v for v in summary.index if v != "real_teacher"]
for i in SENTENCE_IDX + [TECH_IDX]:
    if i >= len(texts):
        continue
    print("=" * 100, f"\nSENTENCE {i}:", texts[i])
    if i < n_held:
        print("REAL TEACHER (not available for the extra sentences)")
        display(Audio(held[i][0]))
    for v in variants:
        print(v)
        display(Audio(str(EVAL_DIR / v / f"{i:02d}.wav")))

## Step 8 - pick the winner and save it
Set `CHOSEN` to the name of the variant that sounded best to you (for example `"step2400_ema"`), or leave `"auto"` to take the suggestion
(the checkpoint with the highest speaker similarity among those whose word error rate is not worse than the baseline by more than 0.05).

In [ ]:
CHOSEN = "auto"

cand = summary.drop(index=[v for v in ("real_teacher", "baseline_no_finetune") if v in summary.index])
base_wer = summary.loc["baseline_no_finetune", "wer"]
ok = cand[cand["wer"] <= (base_wer if pd.isna(base_wer) else base_wer) + 0.05]
suggested = (ok if len(ok) else cand)["speaker_sim"].idxmax()
if CHOSEN == "auto":
    CHOSEN = suggested
assert CHOSEN in cand.index, f"CHOSEN must be one of: {list(cand.index)}"
print("Suggested:", suggested, "| Using:", CHOSEN)

stem, kind = CHOSEN.rsplit("_", 1)
upd = int(stem.replace("step", ""))
src_stem = next(p.stem for p in ckpts if UPD[p.stem] == upd)
src = SLIM / f"{src_stem}_{kind}.safetensors"
FINAL = OUT / "final_model"
shutil.rmtree(FINAL, ignore_errors=True)
FINAL.mkdir(parents=True)
shutil.copy(src, FINAL / "teacher_f5.safetensors")
shutil.copy(data_dir / "vocab.txt", FINAL / "vocab.txt")
shutil.copy(ref_audio_proc, FINAL / "voice_prompt.wav")
(FINAL / "voice_prompt.txt").write_text(ref_text_proc)
(FINAL / "info.json").write_text(json.dumps({
    "base_model": "SWivid/F5-TTS F5TTS_v1_Base (CC-BY-NC)", "f5_tts_commit": F5_COMMIT, "chosen": CHOSEN,
    "use_ema": kind == "ema", "nfe_step": NFE_STEPS, "learning_rate": LEARNING_RATE, "total_updates": TOTAL_UPDATES,
    "train_minutes": round(sum(durations) / 60, 1)}, indent=2))
# keep the samples and the numbers too
shutil.make_archive(str(OUT / "eval_samples"), "zip", EVAL_DIR)
print("\nSaved to", FINAL)
for p in sorted(FINAL.iterdir()):
    print(f"  {p.name:25s} {p.stat().st_size / 1e6:8.1f} MB")
print("Also saved:", OUT / "results_summary.csv", OUT / "results_all_clips.csv", OUT / "eval_samples.zip")
if not DRIVE_OK:       # no Drive: hand the files to the browser so they land in your Downloads folder
    from google.colab import files
    shutil.make_archive(str(ROOT / "teacher_voice_results"), "zip", OUT)
    files.download(str(ROOT / "teacher_voice_results.zip"))
    print("Downloading teacher_voice_results.zip to your computer (allow the download if the browser asks).")

## Step 9 - make the teacher speak anything
`spoken_form` turns digits and symbols into the words the teacher would say (the model was trained on spoken-form text, so it cannot read `%s` or `0x41`).

In [ ]:
def spoken_form(text):
    def hex_(m):
        return " zero X " + " ".join(num2words(int(c)) if c.isdigit() else c.upper() for c in m.group(1)) + " "
    t = re.sub(r"\b0[xX]([0-9a-fA-F]+)\b", hex_, text)
    t = re.sub(r"%\s*([a-zA-Z])\b", lambda m: " percent " + m.group(1).upper() + " ", t)
    t = re.sub(r"(\d+(?:\.\d+)?)\s*%", lambda m: m.group(1) + " percent", t)
    t = re.sub(r"\d+\.\d+|\d+", lambda m: num2words(float(m.group()) if "." in m.group() else int(m.group())).replace("-", " "), t)
    for sym, word in {"&": " and ", "+": " plus ", "=": " equals ", "@": " at ", "#": " hash ", "\\": " backslash ", "*": " star "}.items():
        t = t.replace(sym, word)
    t = " ".join(t.split())
    return re.sub(r"\s+([,.;:?!])", r"\1", t)


_model = load_model(DiT, ARCH, str(FINAL / "teacher_f5.safetensors"), mel_spec_type="vocos", vocab_file=VOCAB,
                    use_ema=json.load(open(FINAL / "info.json"))["use_ema"], device=DEVICE)


def speak(text, out_path="teacher_says.wav"):
    spoken = spoken_form(text)
    print("Speaking:", spoken)
    generate(_model, spoken, Path(out_path))
    display(Audio(str(out_path)))
    return out_path


speak("The program printed 0x41414141 because the format string contained %x, and it took 100% of the attacker's control.")
# speak("Type anything here. Digits and symbols are converted to words automatically.")

## What to send back / do next
1. Send me the table printed in Step 6 (or `results_summary.csv` from your Drive folder `f5_run`) and tell me which variant you liked best by ear, and whether the **accent** sounds Indian now.
2. Download `voice_capstone/f5_run/final_model/` from Drive: that folder is your fine-tuned teacher voice.